# ACUS220 — Laboratorio de escucha: del timbre al espectro

**Actividad experimental independiente · Clase 04 — Clasificación del llanto infantil**  
Universidad Austral de Chile · Acústica Computacional con Python

> **Principio de la actividad:** escuchar → describir → observar → medir → interpretar.

En esta actividad sintetizaremos **dos sonidos que comparten la misma frecuencia fundamental**, pero tienen distinta distribución de amplitudes entre sus armónicos. Primero los escucharemos; después utilizaremos la **transformada de Fourier**, que ya conocemos, para interpretar el centroide y el ancho de banda espectral.

**Objetivos de aprendizaje**

- Describir diferencias auditivas sin consultar previamente los gráficos.
- Reconocer en un espectro la frecuencia fundamental y los armónicos.
- Interpretar físicamente el *centroide espectral* y el *ancho de banda espectral*.
- Distinguir entre descriptores acústicos y causas del llanto: **una característica por sí sola no identifica el motivo de una vocalización**.

**Alcance:** usamos señales sintéticas, no grabaciones del corpus Donate-a-Cry. Este notebook **no modifica** la Clase 03 ni la Clase 04. Recomendamos trabajar con audífonos y un volumen moderado.

## 1. Primero, escuchamos

Antes de ver espectros, reproduciremos dos sonidos, **A** y **B**. Los dos tendrán la misma frecuencia fundamental, duración y amplitud eficaz (RMS). Así podremos concentrarnos en las diferencias de *timbre*.

No es necesario comprender todavía cómo se forman: escucharemos primero y explicaremos después.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

print("NumPy:", np.__version__)
print("Preparación del laboratorio: correcta.")

In [ ]:
# Parámetros compartidos por ambos sonidos
fs = 16_000           # Muestras por segundo (Hz)
duracion = 3.0        # Segundos
f0 = 220.0            # Frecuencia fundamental (Hz)
n_armonicos = 8
rms_objetivo = 0.12   # Amplitud eficaz digital (no equivale a dB SPL)

tiempo = np.arange(int(fs * duracion)) / fs

# Estas amplitudes controlan cuánto aporta cada armónico.
# No es necesario examinarlas aún: primero escucharemos.
amplitudes_A = np.array([1.00, 0.22, 0.09, 0.04, 0.02, 0.01, 0.01, 0.01])
amplitudes_B = np.array([1.00, 0.80, 0.65, 0.55, 0.45, 0.35, 0.30, 0.25])

def rms_digital(senal):
    '''Calcula la amplitud eficaz de una señal digital.'''
    return float(np.sqrt(np.mean(np.asarray(senal, dtype=float) ** 2)))

def sintetizar_sonido(amplitudes):
    '''Suma armónicos, suaviza los extremos e iguala el RMS.'''
    x = np.zeros_like(tiempo)
    for numero_armonico, amplitud in enumerate(amplitudes, start=1):
        x += amplitud * np.sin(2 * np.pi * numero_armonico * f0 * tiempo)

    # Transiciones suaves de 20 ms para evitar clics al comienzo y al final.
    muestras_fundido = int(0.020 * fs)
    rampa = np.sin(np.linspace(0, np.pi / 2, muestras_fundido)) ** 2
    envolvente = np.ones_like(x)
    envolvente[:muestras_fundido] = rampa
    envolvente[-muestras_fundido:] = rampa[::-1]
    x *= envolvente

    return x * (rms_objetivo / rms_digital(x))

sonido_A = sintetizar_sonido(amplitudes_A)
sonido_B = sintetizar_sonido(amplitudes_B)

assert len(sonido_A) == len(sonido_B) == int(fs * duracion)
assert np.isclose(rms_digital(sonido_A), rms_objetivo)
assert np.isclose(rms_digital(sonido_B), rms_objetivo)
assert max(np.max(np.abs(sonido_A)), np.max(np.abs(sonido_B))) < 1.0

print(f"Duración: {duracion:.1f} s")
print(f"Frecuencia fundamental de ambos sonidos: {f0:.0f} Hz")
print(f"Frecuencia de muestreo: {fs} Hz")
print(f"RMS A: {rms_digital(sonido_A):.3f} | RMS B: {rms_digital(sonido_B):.3f}")

In [ ]:
print("SONIDO A — Escúchalo primero")
display(Audio(sonido_A, rate=fs, normalize=False))

print("SONIDO B — Escúchalo después, sin cambiar el volumen")
display(Audio(sonido_B, rate=fs, normalize=False))

### Preguntas de escucha (antes de observar los gráficos)

1. ¿Escuchas una diferencia entre A y B? ¿Cómo la describirías con tus propias palabras?
2. ¿Cuál te parece más **brillante**, **áspero** o **rico en armónicos**?
3. Si ambos sonidos comparten la frecuencia fundamental de 220 Hz, ¿qué podría explicar la diferencia?

**Anota tus impresiones antes de continuar.** No existe una única palabra correcta para describir el timbre: nos interesan las observaciones y las hipótesis.

## 2. Observamos la forma de onda

Una señal acústica digital consiste en muestras de amplitud ordenadas en el tiempo. Una forma de onda permite identificar diferencias temporales, pero no siempre resulta sencillo deducir de ella qué armónicos están presentes.

Mostramos solamente **35 milisegundos** de cada sonido para apreciar sus oscilaciones.

In [ ]:
# Elegimos una ventana central, lejos de los fundidos de entrada y salida.
inicio = int(1.0 * fs)
longitud = int(0.035 * fs)
segmento_t = tiempo[inicio:inicio + longitud]

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(segmento_t * 1000, sonido_A[inicio:inicio + longitud], label="Sonido A", lw=1.6)
ax.plot(segmento_t * 1000, sonido_B[inicio:inicio + longitud], label="Sonido B", lw=1.3, alpha=0.78)
ax.set_xlabel("Tiempo (ms)")
ax.set_ylabel("Amplitud digital")
ax.set_title("Dos formas de onda con la misma frecuencia fundamental")
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

**Observación:** que los sonidos compartan una frecuencia fundamental no significa que tengan la misma forma de onda. La forma final depende de cómo se combinan los armónicos. Para ver esas componentes con mayor claridad, recurrimos a Fourier.

## 3. Observamos sus armónicos con la transformada de Fourier

Recordemos: la transformada de Fourier permite describir una señal a partir de sus componentes frecuenciales.

En nuestro ejemplo, las componentes principales aparecen en los múltiplos de la frecuencia fundamental:

\[
f_k = k f_0, \qquad k=1,2,\ldots,8.
\]

Es decir: 220 Hz, 440 Hz, 660 Hz, …, 1760 Hz.

Tomamos un segundo de señal **en el centro**, sin los fundidos, y calculamos su transformada discreta de Fourier. Elegimos exactamente un segundo para que estas componentes caigan en los bins de la FFT y los gráficos se lean con facilidad.

In [ ]:
# Un segmento de 1 s, situado lejos del comienzo y final del audio.
inicio_fft = int(1.0 * fs)
N = int(1.0 * fs)
ventana_A = sonido_A[inicio_fft:inicio_fft + N]
ventana_B = sonido_B[inicio_fft:inicio_fft + N]

frecuencias_fft = np.fft.rfftfreq(N, d=1/fs)
magnitud_A = (2.0 / N) * np.abs(np.fft.rfft(ventana_A))
magnitud_B = (2.0 / N) * np.abs(np.fft.rfft(ventana_B))

frecuencias_armonicas = f0 * np.arange(1, n_armonicos + 1)
indices_armonicos = [int(np.argmin(np.abs(frecuencias_fft - f))) for f in frecuencias_armonicas]

fig, ax = plt.subplots(figsize=(10, 4))
ancho_barra = 24
ax.bar(frecuencias_armonicas - ancho_barra / 2,
       magnitud_A[indices_armonicos], width=ancho_barra, label="Sonido A")
ax.bar(frecuencias_armonicas + ancho_barra / 2,
       magnitud_B[indices_armonicos], width=ancho_barra, label="Sonido B")
ax.set_xticks(frecuencias_armonicas)
ax.set_xticklabels([str(int(f)) for f in frecuencias_armonicas], rotation=35)
ax.set_xlabel("Frecuencia (Hz)")
ax.set_ylabel("Magnitud de Fourier (escala lineal)")
ax.set_title("Distribución de amplitud entre los ocho armónicos")
ax.grid(axis="y", alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

### ¿Qué encontramos?

- Los dos sonidos tienen una componente fundamental en **220 Hz**.
- Comparten las mismas **frecuencias armónicas**.
- Lo que cambia es la **magnitud relativa** de los armónicos: A concentra más amplitud en los primeros; B conserva una mayor participación de armónicos superiores.

**Pregunta:** a partir del espectro, ¿cuál esperarías que tuviese un centroide espectral más alto? Haz tu predicción antes de calcularlo.

## 4. Centroide espectral: ¿dónde se sitúa el «centro de masa» del espectro?

El **centroide espectral** es una media ponderada de las frecuencias. En esta actividad usamos la **magnitud** de la FFT como peso:

\[
C = \frac{\sum_k f_k\,|X[k]|}{\sum_k |X[k]|}.
\]

Una intuición útil es imaginar el espectro como una distribución de pesos sobre el eje de frecuencias: el centroide indica dónde se equilibraría esa distribución.

**Importante:** el centroide **no es** la frecuencia fundamental. Un espectro puede mantener la misma frecuencia fundamental y desplazar su centroide hacia frecuencias más altas si aumentan los armónicos superiores.

## 5. Ancho de banda espectral: ¿qué tan disperso está el espectro?

El **ancho de banda espectral** que emplearemos aquí se calcula como una dispersión (desviación estándar ponderada) alrededor del centroide:

\[
B = \sqrt{\frac{\sum_k (f_k-C)^2\,|X[k]|}{\sum_k |X[k]|}}.
\]

Un valor mayor indica que, de acuerdo con esta definición, la distribución espectral está más extendida alrededor de su centroide. **No representa un corte rígido** entre una frecuencia inferior y otra superior.

Ambas características se expresan en **Hz**.

In [ ]:
def descriptores_espectrales(frecuencias, magnitudes):
    '''Centroide y ancho de banda: pesos = magnitud del espectro.'''
    pesos = np.asarray(magnitudes, dtype=float)
    suma_pesos = np.sum(pesos)
    if suma_pesos <= 0:
        raise ValueError("El espectro no contiene magnitud positiva.")

    centroide = np.sum(frecuencias * pesos) / suma_pesos
    ancho_banda = np.sqrt(np.sum((frecuencias - centroide)**2 * pesos) / suma_pesos)
    return float(centroide), float(ancho_banda)

centroide_A, ancho_A = descriptores_espectrales(frecuencias_fft, magnitud_A)
centroide_B, ancho_B = descriptores_espectrales(frecuencias_fft, magnitud_B)

print("                   Sonido A        Sonido B")
print(f"Frecuencia f0:     {f0:7.1f} Hz     {f0:7.1f} Hz")
print(f"RMS digital:       {rms_digital(sonido_A):7.3f}        {rms_digital(sonido_B):7.3f}")
print(f"Centroide:         {centroide_A:7.1f} Hz     {centroide_B:7.1f} Hz")
print(f"Ancho de banda:    {ancho_A:7.1f} Hz     {ancho_B:7.1f} Hz")

assert centroide_B > centroide_A
assert ancho_B > ancho_A

### Interpretamos los resultados

El sonido **B** debería presentar un centroide mayor, porque sus armónicos superiores tienen más peso. Con estas amplitudes también esperamos mayor ancho de banda espectral: su distribución se extiende más sobre el eje de frecuencias.

El experimento conecta tres observaciones: **lo escuchado** (diferencias de timbre), **lo visto** (distintas amplitudes armónicas) y **lo calculado** (descriptores en Hz).

**Precaución:** esta relación resulta clara porque controlamos artificialmente la señal. En grabaciones reales hay variaciones temporales, ruido, reverberación y muchos otros factores. Ni un centroide alto ni un ancho de banda grande permiten, por sí solos, explicar *por qué* llora un bebé.

## 6. Dos características adicionales: RMS y ZCR

- **RMS:** resume la amplitud eficaz de la señal digital. Aquí la fijamos en **0,12** para ambos sonidos. En grabaciones no calibradas **no se debe confundir** RMS digital con nivel de presión sonora en dB SPL.
- **ZCR (zero-crossing rate):** cuenta cuántas veces una señal cambia de signo por unidad de tiempo. Puede reaccionar a contenido de altas frecuencias, forma de onda, ruido y desplazamientos de continua. **No es una medida directa de frecuencia fundamental.**

Calculemos un ejemplo simple de ZCR: número de cambios de signo por segundo dentro de un segmento de 1 s.

In [ ]:
def zcr_por_segundo(senal, fs):
    cambios = np.count_nonzero(np.signbit(senal[1:]) != np.signbit(senal[:-1]))
    return cambios / (len(senal) / fs)

print(f"ZCR A (cambios/s): {zcr_por_segundo(ventana_A, fs):.1f}")
print(f"ZCR B (cambios/s): {zcr_por_segundo(ventana_B, fs):.1f}")
print("Recuerda: ZCR es sensible a la forma de onda; no reemplaza el análisis espectral.")

## 7. ¿Y los MFCC? Una primera intuición, sin profundizar todavía

Ya sabemos que dos sonidos pueden diferir en la **forma de su espectro**. Los **MFCC** (*Mel-Frequency Cepstral Coefficients*) buscan resumir esa forma mediante una secuencia de operaciones:

1. Obtenemos información espectral en tramas cortas.
2. Agrupamos la energía en bandas dispuestas según una escala **Mel**, inspirada en la percepción auditiva.
3. Aplicamos un **logaritmo** a las energías por banda.
4. Usamos una **transformada cosenoidal** para obtener una representación compacta de esas energías logarítmicas.

No necesitamos dominar hoy todos sus detalles matemáticos. Es suficiente comprender que **los MFCC son una representación del perfil espectral**, no una medición física individual equivalente a frecuencia fundamental, intensidad o motivo del llanto.

En la Clase 04, las características (RMS, ZCR, centroide, ancho de banda y MFCC) se resumen a lo largo del tiempo mediante estadísticas como la media y la desviación estándar; así cada grabación puede convertirse en un vector de números.

## 8. Frecuencia de muestreo: una decisión acústica que debemos justificar

Este experimento utiliza **16 kHz**. Su frecuencia de Nyquist es **8 kHz**, mientras que el armónico más alto que sintetizamos es:

\[
8\times220 = 1760\ \mathrm{Hz}.
\]

Por lo tanto, los ocho armónicos quedan cómodamente dentro del rango representable. El notebook de clasificación de la Clase 04 propone procesar las grabaciones a **8 kHz**, lo que limita la información espectral representable a frecuencias inferiores a **4 kHz**.

**Que esa decisión sea aceptable para este ejemplo sintético no demuestra que lo sea para todo el llanto infantil.** Debemos evaluar ese compromiso entre contenido acústico disponible y costo computacional cuando volvamos al corpus real.

## 9. Cierre de la experiencia

Responde brevemente con tus palabras:

1. ¿Por qué los sonidos A y B suenan diferentes si comparten una frecuencia fundamental de 220 Hz?
2. ¿En cuál se observan armónicos superiores relativamente más intensos?
3. ¿Qué representa físicamente el centroide espectral? ¿Por qué no es lo mismo que la frecuencia fundamental?
4. ¿Qué mide el ancho de banda utilizado en esta actividad?
5. ¿Qué límites tendría interpretar estas características como si determinaran automáticamente la causa de un llanto infantil?

**Conexión con la Clase 04:** ahora que las características tienen un significado acústico, podremos discutir cómo formar una matriz de características \(X\) y etiquetas \(y\), y cómo evaluar clasificadores de manera reproducible sin confundir exactitud global con capacidad de distinguir todas las clases.

---

*Notebook auxiliar de experimentación para ACUS220. No requiere acceso a Donate-a-Cry ni modifica los notebooks publicados.*